In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

N_SIMS = 100_000
SEED = 42

# Read data
DATA_DIR = Path.cwd() / "testfiles_" / "data"
CSV_PATH = DATA_DIR / "test5_1.csv"

# 1) Read covariance from CSV
df = pd.read_csv(CSV_PATH, header=0)
cols = list(df.columns)
Sigma = df.to_numpy(dtype=float)

# Symmetrize and make Cholesky-friendly by adding small diagonal if needed
Sigma = (Sigma + Sigma.T) / 2 # symmetrize the matrix
n = Sigma.shape[0]
eps = 1e-12
for _ in range(8):
    try:
        L = np.linalg.cholesky(Sigma + eps * np.eye(n))
        Sigma = Sigma + eps * np.eye(n)
        break
    except np.linalg.LinAlgError:
        eps *= 10
else:
    raise RuntimeError("Input covariance is not positive definite; please check the CSV.")

# 2) Simulate X ~ N(0, Sigma)
rng = np.random.default_rng(SEED)
Z = rng.standard_normal(size=(N_SIMS, n))
X = Z @ L.T

# 3) Sample covariance (output)
S_hat = np.cov(X, rowvar=False, ddof=1)

print(", ".join(cols))
for i in range(n):
    row = ", ".join(f"{S_hat[i, j]:.16f}" for j in range(n))
    print(row)

x1, x2, x3, x4, x5
0.0850775302292917, 0.0880555990250735, 0.0425152688281006, 0.0090092577375954, 0.0038866551168811
0.0880555990250735, 0.1614940429002212, 0.0586297716476044, 0.0124056878735872, 0.0053550702635743
0.0425152688281006, 0.0586297716476044, 0.0376732746138158, 0.0059905997108562, 0.0025918232037268
0.0090092577375954, 0.0124056878735872, 0.0059905997108562, 0.0016921836828265, 0.0005481821995909
0.0038866551168811, 0.0053550702635743, 0.0025918232037268, 0.0005481821995909, 0.0003155039377804
